# Banking Data Inventory

This notebook is used to explore and understand all the CSV files in `data/raw/`. We will:
1. Get an overview of all files (shape + first few rows)
2. Run data quality checks (dtypes, missing values, duplicates, cardinality)
3. Find relationships between tables using automatic mapping


In [1]:
import pandas as pd
import numpy as np
import os
from itertools import combinations

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Path to raw data files
raw_path = "../data/raw"


## 1) List all raw CSV files

Let us see what data files we have.

In [2]:
# Find all raw files
files = [f for f in os.listdir(raw_path) if f.endswith(('.csv', '.xlsx', '.xls'))]
files


['accounts.csv',
 'branches.csv',
 'cards.csv',
 'card_transactions.csv',
 'customers.csv',
 'employees.csv',
 'loan_payments.csv']

## 2) Quick inventory: shape and first rows for all files

We create a helper function `show_inventory()` to quickly look at every CSV at once. This gives us the size of each table and a peek at the data.

In [3]:
def show_inventory(files_list=None):
    if files_list is None:
        files_list = sorted([f for f in os.listdir(raw_path) if f.endswith('.csv')])
    for file_name in files_list:
        print('=' * 50)
        print(f'FILE: {file_name}')
        print('=' * 50)
        df = pd.read_csv(os.path.join(raw_path, file_name))
        print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n')
        display(df.head())
        print('\n')


In [ ]:
show_inventory()


FILE: accounts.csv


Shape: 95,000 rows x 7 columns



,account_id,customer_id,branch_id,account_type,balance,open_date,status
0,1,11749,141,Current,95902.00,2019-10-20,Active
1,2,47716,43,Current,13261.61,2026-02-03,Active
2,3,56143,90,Savings,26495.92,2020-10-19,Active
3,4,29193,113,Salary,74682.72,2012-02-22,Active
4,5,49907,28,Current,40419.01,2007-01-03,Active




FILE: branches.csv
Shape: 150 rows x 6 columns



,branch_id,branch_name,city,state,opened_date,ifsc_code
0,1,Pune Branch 1,Pune,Maharashtra,2016-05-23,BNK0000000
1,2,Bhopal Branch 2,Bhopal,Madhya Pradesh,2002-01-13,BNK0000001
2,3,Lucknow Branch 3,Lucknow,Uttar Pradesh,2017-12-10,BNK0000002
3,4,Kolkata Branch 4,Kolkata,West Bengal,2002-05-04,BNK0000003
4,5,Kolkata Branch 5,Kolkata,West Bengal,2005-11-09,BNK0000004




FILE: card_transactions.csv
Shape: 3,000,000 rows x 6 columns



... (5 outputs left out)



FILE: employees.csv
Shape: 1,800 rows x 6 columns



,employee_id,name,branch_id,role,hire_date,salary
0,1,Richard Rodriguez,30,Customer Service,2016-11-09,62479
1,2,Robert Rodriguez,136,Relationship Manager,2010-08-04,50855
2,3,Arjun Rodriguez,35,Customer Service,2020-02-12,94948
3,4,Arjun Singh,146,Loan Officer,2022-02-11,40756
4,5,Robert Joshi,29,Customer Service,2010-05-09,155016




FILE: loan_payments.csv
Shape: 600,000 rows x 7 columns



,payment_id,loan_id,payment_date,amount_paid,principal_component,interest_component,late_payment_flag
0,1,21070,2020-05-06,8144.05,5930.89,2213.16,0
1,2,17959,2011-09-23,6128.01,4406.78,1721.23,0
2,3,7239,2025-02-25,27946.17,25195.76,2750.41,0
3,4,9993,2015-10-04,5715.37,4082.41,1632.96,0
4,5,5487,2017-07-26,10119.72,9474.29,645.43,0


## 3) Data quality checks

Now let us check for data types, missing values, duplicates, categorical cardinality, and basic numeric summaries.

In [5]:
def data_quality_check(file_name):
    print('=' * 70)
    print(f'DATA QUALITY REPORT: {file_name}')
    print('=' * 70)
    df = pd.read_csv(os.path.join(raw_path, file_name))
    print(f'\nShape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    print('\n[ COLUMN DATA TYPES ]')
    display(df.dtypes.to_frame('dtype'))
    print('\n[ MISSING VALUES ]')
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        print('No missing values found!')
    else:
        missing_percent = (missing / len(df) * 100).round(2)
        missing_df = pd.concat([missing, missing_percent], axis=1, keys=['count', '%'])
        display(missing_df)
    print('\n[ DUPLICATES ]')
    dup_count = df.duplicated().sum()
    print(f'Duplicate rows: {dup_count:,}')
    obj_cols = df.select_dtypes(include=['object']).columns
    if len(obj_cols) > 0:
        print('\n[ CATEGORICAL CARDINALITY ]')
        card = df[obj_cols].nunique().sort_values(ascending=False).to_frame('unique_values')
        display(card.head(15))
    print('\n[ NUMERIC SUMMARY ]')
    display(df.describe().T)
    print('\n\n')

files = sorted([f for f in os.listdir(raw_path) if f.endswith('.csv')])
for f in files:
    data_quality_check(f)


DATA QUALITY REPORT: accounts.csv

Shape: 95,000 rows x 7 columns

[ COLUMN DATA TYPES ]


,dtype
account_id,int64
customer_id,int64
branch_id,int64
account_type,object
balance,float64
open_date,object
status,object



[ MISSING VALUES ]
No missing values found!

[ DUPLICATES ]
Duplicate rows: 0

[ CATEGORICAL CARDINALITY ]


,unique_values
open_date,7821
account_type,5
status,3



[ NUMERIC SUMMARY ]


,count,mean,std,min,25%,50%,75%,max
account_id,95000.00,47500.50,27424.28,1.00,23750.75,47500.50,71250.25,95000.00
customer_id,95000.00,29991.21,17264.58,1.00,15114.75,30000.00,44910.00,60000.00
branch_id,95000.00,75.45,43.30,1.00,38.00,75.00,113.00,150.00
balance,95000.00,45761.24,45306.41,500.36,13551.05,31768.65,63337.51,539198.59


... (32 outputs left out)


[ MISSING VALUES ]
No missing values found!

[ DUPLICATES ]
Duplicate rows: 0

[ CATEGORICAL CARDINALITY ]


,unique_values
payment_date,5964



[ NUMERIC SUMMARY ]


,count,mean,std,min,25%,50%,75%,max
payment_id,600000.00,300000.50,173205.23,1.00,150000.75,300000.50,450000.25,600000.00
loan_id,600000.00,11003.13,6343.12,1.00,5516.00,10997.00,16495.00,22000.00
amount_paid,600000.00,10397.46,9856.84,526.27,3409.95,7361.83,14138.02,147281.59
principal_component,600000.00,8487.30,7988.88,500.03,2798.77,6032.06,11571.88,110436.89
interest_component,600000.00,1910.16,2148.62,25.40,505.98,1177.95,2507.29,36844.70
late_payment_flag,600000.00,0.12,0.33,0.00,0.00,0.00,0.00,1.00


## 4) Automatic relationship mapping

We look for connections between tables using exact column overlaps, ID-like columns and uniqueness, and value overlap.

In [6]:
files = sorted([f for f in os.listdir(raw_path) if f.endswith('.csv')])
data = {}
for f in files:
    data[f] = pd.read_csv(os.path.join(raw_path, f))
    print(f'Loaded {f} -> {data[f].shape}')

print('\nAll files loaded!')


Loaded accounts.csv -> (95000, 7)
Loaded branches.csv -> (150, 6)
Loaded card_transactions.csv -> (3000000, 6)
Loaded cards.csv -> (65000, 8)
Loaded customers.csv -> (60000, 12)
Loaded employees.csv -> (1800, 6)
Loaded loan_payments.csv -> (600000, 7)

All files loaded!


In [7]:
print('=== EXACT COLUMN NAME OVERLAPS ===')
found_overlap = False
for f1, f2 in combinations(files, 2):
    common = set(data[f1].columns) & set(data[f2].columns)
    if common:
        found_overlap = True
        print(f'{f1} <-> {f2}: {sorted(list(common))}')
if not found_overlap:
    print('No exact column name overlaps found between any files.')


=== EXACT COLUMN NAME OVERLAPS ===
accounts.csv <-> branches.csv: ['branch_id']
accounts.csv <-> cards.csv: ['account_id', 'customer_id', 'status']
accounts.csv <-> customers.csv: ['customer_id']
accounts.csv <-> employees.csv: ['branch_id']
branches.csv <-> customers.csv: ['city', 'state']
branches.csv <-> employees.csv: ['branch_id']
card_transactions.csv <-> cards.csv: ['card_id']
cards.csv <-> customers.csv: ['customer_id']
customers.csv <-> employees.csv: ['name']


In [8]:
print('=== ID-LIKE COLUMNS (UNIQUENESS CHECK) ===')
for f, df in data.items():
    id_cols = []
    for c in df.columns:
        cl = c.lower()
        if 'id' in cl or cl.endswith('_key') or cl.endswith('key') or cl.endswith('_no') or cl.endswith('number'):
            id_cols.append(c)
    if id_cols:
        print(f'\n{f}:')
        for c in id_cols:
            is_unique = df[c].nunique(dropna=False) == len(df)
            nulls = df[c].isna().sum()
            print(f'  - {c}: {df[c].nunique(dropna=False):,} unique | unique key? {is_unique} | nulls: {nulls}')
    else:
        print(f'\n{f}: No obvious ID-like columns found')


=== ID-LIKE COLUMNS (UNIQUENESS CHECK) ===

accounts.csv:
  - account_id: 95,000 unique | unique key? True | nulls: 0
  - customer_id: 47,640 unique | unique key? False | nulls: 0
  - branch_id: 150 unique | unique key? False | nulls: 0

branches.csv:
  - branch_id: 150 unique | unique key? True | nulls: 0

card_transactions.csv:


  - card_txn_id: 3,000,000 unique | unique key? True | nulls: 0
  - card_id: 65,000 unique | unique key? False | nulls: 0

cards.csv:
  - card_id: 65,000 unique | unique key? True | nulls: 0
  - customer_id: 39,627 unique | unique key? False | nulls: 0
  - account_id: 47,128 unique | unique key? False | nulls: 0

customers.csv:
  - customer_id: 60,000 unique | unique key? True | nulls: 0

employees.csv:
  - employee_id: 1,800 unique | unique key? True | nulls: 0
  - branch_id: 150 unique | unique key? False | nulls: 0

loan_payments.csv:
  - payment_id: 600,000 unique | unique key? True | nulls: 0
  - loan_id: 22,000 unique | unique key? False | nulls: 0
  - amount_paid: 530,881 unique | unique key? False | nulls: 0


In [10]:
print('=== VALUE OVERLAP BETWEEN ID COLUMNS ===')
id_map = {}
for f, df in data.items():
    id_map[f] = []
    for c in df.columns:
        cl = c.lower()
        if 'id' in cl:
            id_map[f].append(c)

for f1, f2 in combinations(files, 2):
    for c1 in id_map[f1]:
        for c2 in id_map[f2]:
            if c1 == c2:
                continue
            s1 = set(data[f1][c1].dropna().astype(str))
            s2 = set(data[f2][c2].dropna().astype(str))
            if len(s1) == 0 or len(s2) == 0:
                continue
            inter = s1.intersection(s2)
            if len(inter) == 0:
                continue
            overlap1 = len(inter) / len(s1) * 100
            overlap2 = len(inter) / len(s2) * 100
            print(f'{f1}.{c1} ({len(s1):,}) <-> {f2}.{c2} ({len(s2):,}) | intersection: {len(inter):,} | {overlap1:.1f}% of {f1} | {overlap2:.1f}% of {f2}')


=== VALUE OVERLAP BETWEEN ID COLUMNS ===
accounts.csv.account_id (95,000) <-> branches.csv.branch_id (150) | intersection: 150 | 0.2% of accounts.csv | 100.0% of branches.csv
accounts.csv.customer_id (47,640) <-> branches.csv.branch_id (150) | intersection: 118 | 0.2% of accounts.csv | 78.7% of branches.csv
accounts.csv.account_id (95,000) <-> card_transactions.csv.card_txn_id (3,000,000) | intersection: 95,000 | 100.0% of accounts.csv | 3.2% of card_transactions.csv
accounts.csv.account_id (95,000) <-> card_transactions.csv.card_id (65,000) | intersection: 65,000 | 68.4% of accounts.csv | 100.0% of card_transactions.csv
accounts.csv.customer_id (47,640) <-> card_transactions.csv.card_txn_id (3,000,000) | intersection: 47,640 | 100.0% of accounts.csv | 1.6% of card_transactions.csv
accounts.csv.customer_id (47,640) <-> card_transactions.csv.card_id (65,000) | intersection: 47,640 | 100.0% of accounts.csv | 73.3% of card_transactions.csv
accounts.csv.branch_id (150) <-> card_transaction

## 5) Summary / Notes

**What we did:**
- Listed all files in `data/raw/`
- Inspected shape and first rows for each table
- Ran a full data quality report (dtypes, missing values, duplicates, cardinality, numeric stats)
- Mapped potential relationships between tables (exact overlaps, ID uniqueness, value overlap)

**Next steps (when we continue):**
- Convert date columns to `datetime` where appropriate
- Investigate any columns that need cleaning based on the reports above
- Build merge plan using the relationship mapping results and start joining tables for deeper analysis in `02_loan_repayment.ipynb`

In [ ]:
# Load the key tables
customers = pd.read_csv(os.path.join(raw_path, "customers.csv"))
accounts = pd.read_csv(os.path.join(raw_path, "accounts.csv"))
branches = pd.read_csv(os.path.join(raw_path, "branches.csv"))

# Merge customer + account
cust_acc = accounts.merge(customers, on="customer_id", how="left", suffixes=("_acc", "_cust"))

# Add branch info
cust_acc_branch = cust_acc.merge(branches, on="branch_id", how="left")
cust_acc_branch.head()
#Step 2: Add cards and transactions (if needed for analysis)
cards = pd.read_csv(os.path.join(raw_path, "cards.csv"))
card_txn = pd.read_csv(os.path.join(raw_path, "card_transactions.csv"))

# cards to accounts/customers
cards_full = cards.merge(cust_acc_branch, on=["customer_id", "account_id"], how="left")

# transactions to cards
txn_full = card_txn.merge(cards_full, on="card_id", how="left")
txn_full.head()

,card_txn_id,card_id,txn_date,merchant_category,amount,is_fraud,customer_id,account_id,card_type,issue_date,expiry_date,credit_limit,status_x,branch_id,account_type,balance,open_date,status_y,name,gender,date_of_birth,city_x,state_x,phone,email,occupation,annual_income,join_date,credit_score,branch_name,city_y,state_y,opened_date,ifsc_code
0,1,52505,2025-01-28,Education,2162.41,0,29618,4572,Credit - Classic,2024-07-26,2029-07-25,0,Active,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,12716,2022-08-26,Education,2236.55,0,31135,44900,Credit - Classic,2018-06-19,2023-06-18,50000,Active,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,49060,2019-10-07,Travel,1184.59,0,29734,45209,Debit,2020-02-28,2025-02-26,50000,Active,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,18295,2019-03-07,Travel,124.79,0,23150,65201,Debit,2023-01-02,2028-01-01,0,Active,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,54342,2026-03-26,Insurance,546.10,0,4878,20873,Credit - Classic,2018-04-15,2023-04-14,0,Blocked,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
